# Analisis Segmentasi Usaha Jasa di Surabaya–Sidoarjo

Notebook ini membersihkan CSV mentah, mengaudit baris bergeser, membangun segmentasi K-Means berbasis rating dan popularitas ulasan, mengevaluasi k=2–6, serta menyimpan tabel hasil. Jalankan berurutan dari atas ke bawah.

> Catatan: CSV mentah tampak memiliki koma tidak ter-escape sehingga sejumlah baris bergeser kolom. Baris dipakai hanya jika rating 0–5, ulasan non-negatif, koordinat masuk akal untuk Jawa Timur, serta sumber/kategori/nama mengidentifikasi usaha jasa laundry, kebersihan, atau perawatan sepatu.


In [ ]:
# Jika belum terpasang, aktifkan baris berikut:
# %pip install pandas numpy scikit-learn matplotlib

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

DATA_PATH = Path("data_raw_umkm")  # sesuaikan jika CSV berada di folder lain
OUT_DIR = Path("hasil_analisis")
OUT_DIR.mkdir(exist_ok=True)
df = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=True)
print("Dimensi data mentah:", df.shape)
display(df.head())


In [ ]:
def to_number(value):
    if pd.isna(value): return np.nan
    s = str(value).strip().replace(",", ".")
    # memperbaiki format koordinat seperti -7.320.130 -> -7.320130
    if s.count(".") > 1:
        parts = s.split(".")
        s = parts[0] + "." + "".join(parts[1:])
    try: return float(s)
    except (TypeError, ValueError): return np.nan

for col in ["rating", "jumlah_ulasan", "latitude", "longitude"]:
    df[col + "_num"] = df[col].map(to_number)

service_pattern = r"laundry|binatu|sepatu|clean|kebersihan|semir"
service_mask = (df["kategori"].fillna("").str.contains(service_pattern, case=False, regex=True) |
                 df["nama_usaha"].fillna("").str.contains(service_pattern, case=False, regex=True) |
                 df["sumber_file"].eq("Scraping_Jasa"))
valid_mask = (df["nama_usaha"].notna() & df["kategori"].notna() &
              df["rating_num"].between(0, 5) & df["jumlah_ulasan_num"].ge(0) &
              df["latitude_num"].between(-8, -6) & df["longitude_num"].between(110, 116))

print({"baris_mentah": len(df), "kandidat_jasa": int(service_mask.sum()),
       "kandidat_valid": int((service_mask & valid_mask).sum()),
       "kandidat_gagal_validasi": int((service_mask & ~valid_mask).sum())})


In [ ]:
clean = df.loc[service_mask & valid_mask].copy()
clean["kota_norm"] = clean["kota"].fillna("").str.lower().map(
    lambda s: "Sidoarjo" if "sidoarjo" in s else ("Surabaya" if "surabaya" in s else "Tidak diketahui"))
addr = clean["alamat"].fillna("").str.lower()
clean.loc[(clean.kota_norm == "Tidak diketahui") & addr.str.contains("sidoarjo|waru|wedoro|tropodo|sedati"), "kota_norm"] = "Sidoarjo"
clean.loc[(clean.kota_norm == "Tidak diketahui") & addr.str.contains("surabaya"), "kota_norm"] = "Surabaya"
clean["name_norm"] = clean["nama_usaha"].str.lower().str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()
clean = clean.sort_values("jumlah_ulasan_num", ascending=False).drop_duplicates(
    subset=["name_norm", "latitude_num", "longitude_num"], keep="first").copy()
print("Baris setelah filter dan deduplikasi:", len(clean))
print("Listing rating 0 & ulasan 0 (dipisah):", int(((clean.rating_num == 0) & (clean.jumlah_ulasan_num == 0)).sum()))
print("Listing tanpa rating/ulasan:", int(((clean.rating_num == 0) & (clean.jumlah_ulasan_num == 0)).sum()))
print("Kota:"); display(clean.kota_norm.value_counts(dropna=False).to_frame("jumlah"))
print("Kategori teratas:"); display(clean.kategori.value_counts().head(15).to_frame("jumlah"))


In [ ]:
# Rating merepresentasikan kepuasan agregat; log(1 + jumlah ulasan) meredam kemencengan popularitas.
# Lokasi tidak dimasukkan ke jarak K-Means agar segmentasi perilaku tidak didominasi koordinat.
no_review = clean["rating_num"].eq(0) & clean["jumlah_ulasan_num"].eq(0)
cluster_data = clean.loc[~no_review].copy()
X = cluster_data[["rating_num"]].copy()
X["log_jumlah_ulasan"] = np.log1p(cluster_data["jumlah_ulasan_num"])
scaler = StandardScaler()
Z = scaler.fit_transform(X)

metric_rows = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(Z)
    metric_rows.append({"k": k, "inertia": km.inertia_,
        "silhouette": silhouette_score(Z, labels),
        "davies_bouldin": davies_bouldin_score(Z, labels),
        "calinski_harabasz": calinski_harabasz_score(Z, labels)})
metrics = pd.DataFrame(metric_rows)
display(metrics.round(4))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(metrics.k, metrics.inertia, marker="o"); ax[0].set(title="Elbow / inertia", xlabel="Jumlah cluster (k)", ylabel="Inertia")
ax[1].plot(metrics.k, metrics.silhouette, marker="o", label="Silhouette")
ax[1].set(title="Silhouette per k", xlabel="Jumlah cluster (k)", ylabel="Silhouette", xticks=metrics.k)
plt.tight_layout(); plt.show()


In [ ]:
# Aturan utama: pilih silhouette tertinggi; DBI dipakai sebagai pemeriksaan pendamping.
best_k = int(metrics.sort_values(["silhouette", "davies_bouldin"], ascending=[False, True]).iloc[0].k)
final_model = KMeans(n_clusters=best_k, random_state=42, n_init=20)
cluster_data["cluster"] = final_model.fit_predict(Z)
print("k terpilih:", best_k)
profile = cluster_data.groupby("cluster").agg(
    jumlah_usaha=("nama_usaha", "size"),
    rating_rata2=("rating_num", "mean"),
    rating_median=("rating_num", "median"),
    ulasan_median=("jumlah_ulasan_num", "median"),
    ulasan_rata2=("jumlah_ulasan_num", "mean"),
    ulasan_total=("jumlah_ulasan_num", "sum"),
    porsi_surabaya=("kota_norm", lambda s: (s == "Surabaya").mean()),
    porsi_sidoarjo=("kota_norm", lambda s: (s == "Sidoarjo").mean())
).reset_index()
display(profile.round(3))


In [ ]:
# Plot profil hasil pada skala asli
fig, ax = plt.subplots(figsize=(8, 5))
for cl, g in cluster_data.groupby("cluster"):
    ax.scatter(g["rating_num"], np.log1p(g["jumlah_ulasan_num"]), alpha=.65, label=f"Cluster {cl}")
ax.set(xlabel="Rating", ylabel="log(1 + jumlah ulasan)", title="Profil klaster")
ax.legend(); plt.tight_layout(); plt.show()

metrics.to_csv(OUT_DIR / "metrik_pemilihan_k.csv", index=False)
profile.to_csv(OUT_DIR / "profil_cluster.csv", index=False)
cluster_data.to_csv(OUT_DIR / "data_bersih_dengan_cluster.csv", index=False)
clean.loc[no_review].to_csv(OUT_DIR / "listing_tanpa_ulasan.csv", index=False)
print("Hasil tersimpan di", OUT_DIR.resolve())


## Cara membaca hasil

- Silhouette mendekati 1 berarti pemisahan lebih baik; mendekati 0 berarti klaster bertumpang tindih; nilai negatif menandai banyak objek lebih dekat ke klaster lain.
- DBI lebih kecil biasanya lebih baik; Calinski–Harabasz lebih besar biasanya lebih baik. Jangan memilih k dari satu metrik saja.
- Karena hanya rating dan log jumlah ulasan yang tersedia dan dipakai, label klaster adalah segmen reputasi/popularitas, bukan ukuran UMKM, omzet, kualitas kausal, atau kelayakan bantuan.
- Listing dengan rating 0 dan jumlah ulasan 0 dipisahkan sebagai belum memiliki umpan balik dan tidak ikut membentuk klaster utama.
